In [ ]:
!unzip "/content/drive/MyDrive/Figaro1k.zip"
!mv Figaro1k/Original/Training/* "Figaro1k/Original/"
!rmdir "Figaro1k/Original/Training"
!mv Figaro1k/Original/Testing/* "Figaro1k/Original/"
!rmdir "Figaro1k/Original/Testing"
!mv Figaro1k/GT/Training/* "Figaro1k/GT/"
!rmdir "Figaro1k/GT/Training"
!mv Figaro1k/GT/Testing/* "Figaro1k/GT/"
!rmdir "Figaro1k/GT/Testing"

In [ ]:
import os
from skimage import io
import pickle
from skimage.transform import resize
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
images_path = "Figaro1k/Original/"
masks_path = "Figaro1k/GT/"
images_paths = os.listdir("Figaro1k/Original")
masks_paths = os.listdir("Figaro1k/GT")

In [ ]:
images,masks = [],[]
size = min(len(images_paths),len(masks_paths))
for i in range(size):
    file = images_paths[i].replace('-org.jpg','')
    img_path = file + '-org.jpg'
    mask_path = file + '-gt.pbm'
    if img_path in images_paths and mask_path in masks_paths:
            images.append(io.imread(images_path + img_path,plugin='matplotlib'))
            masks.append( io.imread(masks_path + mask_path,plugin='matplotlib'))

In [ ]:
print("Actual data size:",len(images),len(masks))

Actual data size: 1050 1050


In [ ]:
size = 1050
X = np.zeros((size,224,224,3))
for j in range(size):
  for i in range(3):
      images[j][:,:,i] = images[j][:,:,i] * masks[j][:,:,i] * masks[j][:,:,i]
  non_black_pixels = np.any(images[j] != 0, axis=-1)

  # find the bounding box of the non-black pixels
  non_black_pixels_y = np.any(non_black_pixels, axis=1)
  non_black_pixels_x = np.any(non_black_pixels, axis=0)
  min_x, max_x = np.where(non_black_pixels_x)[0][[0, -1]]
  min_y, max_y = np.where(non_black_pixels_y)[0][[0, -1]]

  # crop the image
  images[j] = images[j][min_y:max_y+1, min_x:max_x+1, :]

  images[j] = resize(images[j], (224, 224))
  X[j] = images[j]

with open(f'/content/drive/MyDrive/hair.pickle', 'wb') as f:
        pickle.dump(X, f)

In [ ]:
pickle_off = open (f'/content/drive/MyDrive/hair.pickle', "rb")
X = pickle.load(pickle_off)

In [ ]:
class_list = [
    {'class': 'straight', 'start_frame': 1, 'end_frame': 150},
    {'class': 'wavy', 'start_frame': 151, 'end_frame': 300},
    {'class': 'curly', 'start_frame': 301, 'end_frame': 450},
    {'class': 'kinky', 'start_frame': 451, 'end_frame': 600},
    {'class': 'braids', 'start_frame': 601, 'end_frame': 750},
    {'class': 'dreadlocks', 'start_frame': 751, 'end_frame': 900},
    {'class': 'short-men', 'start_frame': 901, 'end_frame': 1050}
]

df = pd.DataFrame(columns=['class'])

for class_dict in class_list:
    for frame in range(class_dict['start_frame'], class_dict['end_frame']+1):
        df = df.append({'class': class_dict['class']}, ignore_index=True)
Y = df["class"]
Y = pd.get_dummies(Y)

In [ ]:
from sklearn.model_selection import train_test_split

# split data into training and validation sets
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.15, random_state=42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras import layers, models, callbacks

# Define the input shape
input_shape = (224, 224, 3)

# Create a sequential model
model = models.Sequential()

model.add(layers.Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu', input_shape=input_shape))
model.add(layers.MaxPooling2D(pool_size=(2,2)))
model.add(layers.BatchNormalization())

model.add(layers.Conv2D(filters=128, kernel_size=(3,3), padding='same', activation='relu'))
model.add(layers.MaxPooling2D(pool_size=(2,2)))
model.add(layers.BatchNormalization())

# model.add(layers.Conv2D(filters=256, kernel_size=(3,3), padding='same', activation='relu'))
# model.add(layers.MaxPooling2D(pool_size=(2,2)))
# model.add(layers.BatchNormalization())

# model.add(layers.Conv2D(filters=1028, kernel_size=(5,5), padding='same', activation='relu'))
# model.add(layers.MaxPooling2D(pool_size=(2,2)))
# model.add(layers.BatchNormalization())

# Add the rest of the CNN layers
model.add(layers.Flatten())
# model.add(layers.Dense(units=512, activation='relu'))
# # model.add(layers.Dropout(rate=0.5))
model.add(layers.Dense(units=256, activation='relu'))
# model.add(layers.Dropout(rate=0.5))
model.add(layers.Dense(units=7, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

checkpoint_path = "content/drive/MyDrive/HairClassifier.h5"

checkpoint_callback = callbacks.ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', mode='max', save_best_only=True, save_weights_only=False, save_freq='epoch')

model.fit(X_train, Y_train, epochs=10, validation_data=(X_val, Y_val), batch_size=32, callbacks=[checkpoint_callback])

In [ ]:
from tensorflow.keras.models import load_model


model = load_model("content/drive/MyDrive/HairClassifier.h5")